# Подготовим данные для обучения моделей

## Здесь мы добавляем новые признаки, нормализуем значения и сохраняем готовый для использования моделями датасет.

In [ ]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

In [159]:
# открытие данных
home_dir = 'C:/LinuxShared/магистратура/хакатон2/hackathon/'
customers_df= pd.read_csv(home_dir + 'customers.csv')
geolocation_df= pd.read_csv(home_dir + 'geolocation.csv')
items_df= pd.read_csv(home_dir + 'orders_items.csv')
payments_df= pd.read_csv(home_dir + 'order_payments.csv')
reviews_df= pd.read_csv(home_dir + 'order_reviews.csv')
orders_df= pd.read_csv(home_dir + 'orders.csv')
products_df= pd.read_csv(home_dir + 'products.csv')
sellers_df= pd.read_csv(home_dir + 'sellers.csv')
category_translation_df= pd.read_csv(home_dir + 'product_category_name_translation.csv')

combined_df= pd.read_csv('sellers_target.csv')

C:\Users\днс\AppData\Local\Temp\ipykernel_33764\1096173814.py:5: DtypeWarning:

Columns (1,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.



In [160]:
reviews_df.columns

Index(['Unnamed: 0', 'review_id', 'order_id', 'review_score',
       'review_comment_title', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp'],
      dtype='object')

In [161]:
combined_df.columns
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29111 entries, 0 to 29110
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         29111 non-null  int64  
 1   seller_id          29111 non-null  object 
 2   order_id           29111 non-null  object 
 3   product_id         29111 non-null  object 
 4   price              29111 non-null  float64
 5   freight_value      29111 non-null  float64
 6   customer_id        29111 non-null  object 
 7   order_approved_at  29111 non-null  object 
 8   month              29111 non-null  int64  
 9   year               29111 non-null  int64  
 10  lost_seller        29111 non-null  int64  
dtypes: float64(2), int64(4), object(5)
memory usage: 2.4+ MB


In [163]:
# Мерджим combined_df и reviews_df по ключу order_id
combined_df = pd.merge(combined_df, reviews_df[['order_id', 'review_score']], on='order_id', how='left')

# # Удаляем строки с пропущенными значениями в колонке review_score
# combined_df = combined_df.dropna(subset=['review_score'])

# Проверим результат
print(combined_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29268 entries, 0 to 29267
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         29268 non-null  int64  
 1   seller_id          29268 non-null  object 
 2   order_id           29268 non-null  object 
 3   product_id         29268 non-null  object 
 4   price              29268 non-null  float64
 5   freight_value      29268 non-null  float64
 6   customer_id        29268 non-null  object 
 7   order_approved_at  29268 non-null  object 
 8   month              29268 non-null  int64  
 9   year               29268 non-null  int64  
 10  lost_seller        29268 non-null  int64  
 11  review_score       29072 non-null  float64
dtypes: float64(3), int64(4), object(5)
memory usage: 2.7+ MB
None


In [165]:
df = combined_df.copy()


# Шаг 1: Преобразуем колонку order_approved_at в формат datetime
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])

# Шаг 2: Группируем по продавцам и находим минимальную и максимальную дату заказа
periods = df.groupby(['seller_id']).agg(min_date=('order_approved_at', 'min'),
                                      max_date=('order_approved_at', 'max')).reset_index()

# Шаг 3: Для каждого продавца делим период на 3 равных интервала по 30 дней
results = []
for _, row in periods.iterrows():
    seller_id = row['seller_id']
    min_date = row['min_date']
    max_date = row['max_date']
    
    # Определяем интервалы по 30 дней
    period1_end = min_date + pd.DateOffset(days=30)
    period2_end = min_date + pd.DateOffset(days=60)
    
    # Подсчитываем количество заказов и сумму freight_value для каждого периода
    orders_period1 = df[(df['seller_id'] == seller_id) & (df['order_approved_at'] >= min_date) & (df['order_approved_at'] < period1_end)]
    orders_period2 = df[(df['seller_id'] == seller_id) & (df['order_approved_at'] >= period1_end) & (df['order_approved_at'] < period2_end)]
    orders_period3 = df[(df['seller_id'] == seller_id) & (df['order_approved_at'] >= period2_end) & (df['order_approved_at'] <= max_date)]
    
    # Суммируем количество заказов и freight_value
    result = {
        'seller_id': seller_id,
        'orders_first_30_days': len(orders_period1),
        'freight_value_first_30_days': orders_period1['freight_value'].sum(),
        'orders_second_30_days': len(orders_period2),
        'freight_value_second_30_days': orders_period2['freight_value'].sum(),
        'orders_last_30_days': len(orders_period3),
        'freight_value_last_30_days': orders_period3['freight_value'].sum(),
        
        # Средние значения review_score и price
        'avg_review_score_first_30_days': orders_period1['review_score'].mean(),
        'avg_price_first_30_days': orders_period1['price'].mean(),
        'avg_review_score_second_30_days': orders_period2['review_score'].mean(),
        'avg_price_second_30_days': orders_period2['price'].mean(),
        'avg_review_score_last_30_days': orders_period3['review_score'].mean(),
        'avg_price_last_30_days': orders_period3['price'].mean()
    }
    results.append(result)

# Создаем итоговый датафрейм
aggregated_df = pd.DataFrame(results)

# Выводим результат
print(aggregated_df)

                             seller_id  orders_first_30_days  \
0     001cca7ae9ae17fb1caed9dfb1094831                     5   
1     002100f778ceb8431b7a1020ff7ab48f                    16   
2     004c9cd9d87a3c30c522c48c4fc07416                     7   
3     00720abe85ba0859807595bbf045a33b                     4   
4     00ab3eff1b5192e5f1a63bcecfee11c8                     1   
...                                ...                   ...   
2026  ffc470761de7d0232558ba5e786e57b7                     3   
2027  ffdd9f82b9a447f6f8d4b91554cc7dd3                     1   
2028  ffeee66ac5d5a62fe688b9d26f83f534                     5   
2029  fffd5413c0700ac820c7069d66d98c89                     8   
2030  ffff564a4f9085cd26170f4732393726                     6   

      freight_value_first_30_days  orders_second_30_days  \
0                          325.01                      0   
1                          215.94                     10   
2                          135.19                  

In [166]:
# Мерджим aggregated_df и combined_df по ключу seller_id
aggregated_df = pd.merge(aggregated_df, combined_df[['seller_id', 'lost_seller']], on='seller_id', how='left')

aggregated_df['seller_id'].loc[aggregated_df['lost_seller'] == 1].value_counts()

seller_id
bf84056e679dbe9c69929847a40e338f    40
8cc6a0e5738e61a87b03c78b2ba9db4b    26
0b36063d5818f81ccb94b54adfaebbf5    15
9198786624eaeb375793215cad26cfa6    15
8a87611c08849ffeeccab52aa798b6c7    14
                                    ..
0b46f784306be7200ca1700aa55d819f     1
00d8b143d12632bad99c0ad66ad52825     1
0b1ca3ef18a63d7eb0c8897fa0849c08     1
0b09101900100c0e9d312861fad5a1b9     1
00ab3eff1b5192e5f1a63bcecfee11c8     1
Name: count, Length: 310, dtype: int64

In [167]:
# Добавим признаки изменения наших показателей между последними активными периодами клиентов-продавцов
aggregated_df['delta_orders_second'] = aggregated_df['orders_last_30_days']/aggregated_df['orders_second_30_days']
aggregated_df['delta_orders_first'] = aggregated_df['orders_second_30_days']/aggregated_df['orders_first_30_days']

aggregated_df['delta_score_second'] = aggregated_df['avg_review_score_last_30_days'] - aggregated_df['avg_review_score_second_30_days']
aggregated_df['delta_score_first'] = aggregated_df['avg_review_score_second_30_days'] - aggregated_df['avg_review_score_first_30_days']

aggregated_df['delta_freight_first'] = aggregated_df['freight_value_second_30_days']/aggregated_df['freight_value_first_30_days']
aggregated_df['delta_freight_second'] = aggregated_df['freight_value_last_30_days']/aggregated_df['freight_value_second_30_days']

aggregated_df['delta_price_first'] = aggregated_df['avg_price_second_30_days']/aggregated_df['avg_price_first_30_days']
aggregated_df['delta_price_second'] = aggregated_df['avg_price_last_30_days']/aggregated_df['avg_price_second_30_days']

In [168]:
aggregated_df.columns

Index(['seller_id', 'orders_first_30_days', 'freight_value_first_30_days',
       'orders_second_30_days', 'freight_value_second_30_days',
       'orders_last_30_days', 'freight_value_last_30_days',
       'avg_review_score_first_30_days', 'avg_price_first_30_days',
       'avg_review_score_second_30_days', 'avg_price_second_30_days',
       'avg_review_score_last_30_days', 'avg_price_last_30_days',
       'lost_seller', 'delta_orders_second', 'delta_orders_first',
       'delta_score_second', 'delta_score_first', 'delta_freight_first',
       'delta_freight_second', 'delta_price_first', 'delta_price_second'],
      dtype='object')

In [169]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Колонки для нормализации (исключаем строковые и категориальные колонки)
columns_to_normalize = [
    'orders_first_30_days', 'freight_value_first_30_days',
    'orders_second_30_days', 'freight_value_second_30_days',
    'orders_last_30_days', 'freight_value_last_30_days',
    'avg_review_score_first_30_days', 'avg_price_first_30_days',
    'avg_review_score_second_30_days', 'avg_price_second_30_days',
    'avg_review_score_last_30_days', 'avg_price_last_30_days',
    'delta_orders_second', 'delta_orders_first', 'delta_score_second',
    'delta_score_first', 'delta_freight_first', 'delta_freight_second',
    'delta_price_first', 'delta_price_second'
]

# Шаг 1: Удали строки с бесконечными значениями
aggregated_df = aggregated_df.replace([np.inf, -np.inf], np.nan).dropna(subset=columns_to_normalize)

# Мин-макс нормализация
scaler = MinMaxScaler()

# Применяем нормализацию
aggregated_df[columns_to_normalize] = scaler.fit_transform(aggregated_df[columns_to_normalize])

# Округляем значения до 3 знаков после запятой
aggregated_df[columns_to_normalize] = aggregated_df[columns_to_normalize].round(3)

# Удаляем колонку seller_id
aggregated_df = aggregated_df.drop(columns=['seller_id'])

# Проверим результат
print(aggregated_df.head())

    orders_first_30_days  freight_value_first_30_days  orders_second_30_days  \
7                  0.069                        0.046                  0.052   
8                  0.069                        0.046                  0.052   
9                  0.069                        0.046                  0.052   
10                 0.069                        0.046                  0.052   
11                 0.069                        0.046                  0.052   

    freight_value_second_30_days  orders_last_30_days  \
7                          0.021                0.004   
8                          0.021                0.004   
9                          0.021                0.004   
10                         0.021                0.004   
11                         0.021                0.004   

    freight_value_last_30_days  avg_review_score_first_30_days  \
7                        0.006                           0.734   
8                        0.006              

In [170]:
aggregated_df.to_csv('train_test.csv')